# Lecture 10: Sparse direct solvers

## Recap of the previous part

- Distributed memory for huge dense matrices
- Sparse matrix formats (COO, CSR, CSC)
- Matrix-by-vector product
- Inefficiency of sparse matrix processing
- Approaches to reduce cache misses

## Plan for this week: how to solve large sparse linear systems

- Direct methods (today and the next lecture)
    - LU / Cholesky decomposition
    - Reordering techniques to minimize fill-in
- Krylov methods

## Plan for today


- sparse LU / Cholesky and **fill-in**
- graph view of elimination: the *elimination game*
- orderings
- separators and **nested dissection**
- spectral graph partitioning (Fiedler vector) in detail

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.sparse.csgraph import reverse_cuthill_mckee, connected_components, laplacian
import matplotlib.pyplot as plt
import networkx as nx
%matplotlib inline

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True, linewidth=120)

def lap1d(n):
    return sp.diags_array([-np.ones(n - 1), 2 * np.ones(n), -np.ones(n - 1)], offsets=[-1, 0, 1], format='csr')

def lap2d(n):
    T, I = lap1d(n), sp.eye_array(n)
    return (sp.kron(T, I) + sp.kron(I, T)).tocsc()   # grid node (i, j) has index i*n + j

def lu_nnz(A, perm=None, permc_spec='NATURAL'):
    '''nnz(L) + nnz(U) for LU of P A P^T: symmetric permutation, no row pivoting.'''
    A = sp.csc_array(A)
    if perm is not None:
        A, permc_spec = A[perm, :][:, perm], 'NATURAL'
    lu = spla.splu(sp.csc_matrix(A), permc_spec=permc_spec, diag_pivot_thresh=0.0,
                   options=dict(SymmetricMode=True))
    return lu.L.nnz + lu.U.nnz, lu

## Direct methods for sparse matrices: LU decomposition

- Why can sparse systems be solved faster than in $\mathcal{O}(n^3)$?
- In $A = LU$ (or $A = LL^\top$ for $A = A^\top \succ 0$) the factors $L$ and $U$ can also be sparse.
- Solving with **sparse** triangular factors costs $\mathcal{O}(\mathrm{nnz}(L) + \mathrm{nnz}(U))$.

**Never compute $A^{-1}$: the inverse of a sparse matrix is (generically) dense!**

Counterintuitive bonus: dense $\neq$ structureless. For an irreducible tridiagonal $A$, *every* off-diagonal block of $A^{-1}$ has rank **exactly 1** (semiseparable matrices). This is the starting point of $\mathcal{H}$-matrix / HSS solvers.

In [ ]:
n = 64
A = (lap1d(n) + sp.diags_array(rng.uniform(0, 1, n))).toarray()   # tridiag(-1, 2 + r_i, -1)
Ainv = np.linalg.inv(A)
s = np.linalg.svd(Ainv[: n // 2, n // 2 :], compute_uv=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].spy(A, markersize=2); ax[0].set_title(f'A: nnz = {np.count_nonzero(A)}')
im = ax[1].imshow(np.log10(np.abs(Ainv))); plt.colorbar(im, ax=ax[1])
ax[1].set_title(f'log10 |A^-1|: nnz = {np.count_nonzero(Ainv)} of {n*n}')
ax[2].semilogy(s / s[0], 'o'); ax[2].set_title('singular values of block A^-1[:n/2, n/2:]')
plt.tight_layout(); plt.show()
print('sigma_k / sigma_1 =', s[:4] / s[0])

## And the factors...

In the tridiagonal case without pivoting, $L$ and $U$ are **bidiagonal**: no fill-in at all, $\mathcal{O}(n)$ cost (Thomas algorithm).

For $A = \mathrm{tridiag}(-1, 2, -1)$: $\;u_{kk} = \frac{k+1}{k}$, $\;l_{k+1,k} = -\frac{k}{k+1}$.

In SciPy: `permc_spec='NATURAL'` keeps the column order, `diag_pivot_thresh=0` forbids row pivoting.

In [ ]:
n = 7
A = lap1d(n).tocsc()
lu = spla.splu(A, permc_spec='NATURAL', diag_pivot_thresh=0.0)
print(lu.L.toarray())
print('theory: l_{k+1,k} =', np.array([-k / (k + 1) for k in range(1, n)]))

The SciPy defaults (`COLAMD` column ordering + threshold partial pivoting) are designed for general unsymmetric matrices and need not preserve the bidiagonal structure:

In [ ]:
A = lap1d(n).tocsc()
lu_def = spla.splu(A)
print(lu_def.L.toarray())
print('perm_c =', lu_def.perm_c, ' perm_r =', lu_def.perm_r)
print('nnz(L)+nnz(U): natural =', lu.L.nnz + lu.U.nnz, ' default =', lu_def.L.nnz + lu_def.U.nnz)

## 2D case

For the 2D Laplacian on an $n \times n$ grid ($N = n^2$ unknowns) in natural (row-by-row) ordering, $A$ is banded with bandwidth $n$, and elimination fills the whole band:

$$\mathrm{nnz}(L) \approx N^{3/2}, \qquad \text{flops} \approx N^2.$$

With a good ordering (nested dissection): $\mathrm{nnz}(L) = \mathcal{O}(N \log N)$, flops $= \mathcal{O}(N^{3/2})$, and this is asymptotically optimal for 2D grids (Hoffman, Martin, Rose 1973).

In [ ]:
n = 64
A = lap2d(n)
_, lu_nat = lu_nnz(A)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].spy(A, markersize=0.3); ax[0].set_title(f'A (2D Laplacian), nnz = {A.nnz}')
ax[1].spy(lu_nat.L, markersize=0.3); ax[1].set_title(f'L, natural ordering, nnz = {lu_nat.L.nnz}')
plt.show()

## Main challenge: how to make the factors $L$ and $U$ as sparse as possible?

**Fill-in**: entries that change from zero to nonzero during elimination.

Fill-in depends on the ordering, so before factorization we look for a permutation that produces small fill-in.

**Example**

$$A = \begin{bmatrix} * & * & * & * & *\\ * & * & 0 & 0 & 0 \\ * & 0 & * & 0 & 0 \\ * & 0 & 0 & * & 0 \\ * & 0 & 0 & 0 & * \end{bmatrix}$$

Eliminating from top to bottom gives a dense matrix; eliminating from bottom to top keeps it sparse.

## Dense factors from a sparse matrix, and how a permutation fixes it

$$
\begin{bmatrix} * & * & * & * \\ * & * &  &  \\ * &  & * &  \\ * &  &  & * \end{bmatrix} =
\begin{bmatrix} * &  &  &  \\ * & * &  &  \\ * & * & * &  \\ * & * & * & * \end{bmatrix}
\begin{bmatrix} * & * & * & * \\  & * & * & * \\  &  & * & * \\  &  &  & * \end{bmatrix}
$$

We factorize $PAP^\top$ instead, with $P$ the reversal permutation:

$$
PAP^\top = \begin{bmatrix} * &  &  & * \\  & * &  & * \\  &  & * & * \\ * & * & * & * \end{bmatrix} =
\begin{bmatrix} * &  &  &  \\  & * &  &  \\  &  & * &  \\ * & * & * & * \end{bmatrix}
\begin{bmatrix} * &  &  & * \\  & * &  & * \\  &  & * & * \\  &  &  & * \end{bmatrix}
$$

- The **arrowhead** form (dense row/column *last*) gives sparse factors.
- Same nonzeros, different numbering: $\mathcal{O}(n^2)$ vs $\mathcal{O}(n)$ memory. Let us measure it.

In [ ]:
def arrowhead(n):
    A = sp.lil_array((n, n))
    A[0, :] = 1.0; A[:, 0] = 1.0
    A.setdiag(float(n))          # diagonally dominant => SPD
    return A.tocsc()

for n in [250, 500, 1000, 2000]:
    A, rev = arrowhead(n), np.arange(n)[::-1]
    t0 = time.perf_counter(); bad, _ = lu_nnz(A)
    t1 = time.perf_counter(); good, _ = lu_nnz(A, perm=rev); t2 = time.perf_counter()
    print(f'n = {n:5d} | dense row first: nnz = {bad:8d} ({t1 - t0:.3f} s) | dense row last: nnz = {good:5d} ({t2 - t1:.4f} s)')

A = arrowhead(12)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].spy(A); ax[0].set_title('A')
ax[1].spy(lu_nnz(A)[1].L); ax[1].set_title('L, original order')
ax[2].spy(lu_nnz(A, perm=np.arange(12)[::-1])[1].L); ax[2].set_title('L, reversed order')
plt.show()

## Block version of the arrowhead structure

$$
PAP^\top = \begin{bmatrix} A_{11} &  & A_{13} \\  & A_{22} & A_{23} \\ A_{31} & A_{32} & A_{33}\end{bmatrix}
= \begin{bmatrix} A_{11} & 0 & 0 \\ 0 & A_{22} & 0 \\ A_{31} & A_{32} & S \end{bmatrix}
\begin{bmatrix} I & 0 & A_{11}^{-1}A_{13} \\ 0 & I & A_{22}^{-1}A_{23} \\ 0 & 0 & I\end{bmatrix},
$$


$$S = A_{33} - A_{31}A_{11}^{-1}A_{13} - A_{32}A_{22}^{-1}A_{23} \quad \text{(Schur complement)}.$$

- The problem reduces to **independent** smaller problems with $A_{11}$ and $A_{22}$ (parallelism for free!) plus one small, typically dense, system with $S$.
- Applying the same idea recursively to $A_{11}$, $A_{22}$ gives **nested dissection**.

## Main tool to analyze sparsity of the factors: graph theory

For a symmetric pattern, the graph $G(A)$ has vertices $1, \dots, n$ and an edge $\{i, j\}$ iff $a_{ij} \neq 0$ (the matrix is the adjacency matrix of the graph).

**Elimination game** (Parter 1961, Rose 1970). Eliminating variable $k$ means:
1. delete vertex $k$ from the graph,
2. connect all its remaining neighbours into a **clique**.

New edges are exactly the fill-in, and $\mathrm{nnz}(L) = \sum_k (1 + d_k)$, where $d_k$ is the degree of $k$ at the moment it is eliminated.

**Fill-path theorem** (Rose, Tarjan, Lueker 1976): for $i > j$, $l_{ij} \neq 0$ iff $G(A)$ has a path from $i$ to $j$ whose interior vertices are all numbered lower than $j$.




### Example

*star graph = arrowhead:* eliminating the centre first turns all the leaves into a clique, so $L$ is dense; eliminating the centre last creates no fill at all.

Graphs of $\begin{bmatrix} * & * & * & * \\ * & * &  &  \\ * &  & * &  \\ * &  &  & * \end{bmatrix}$ and $\begin{bmatrix} * &  &  & *  \\  & * &  & * \\  &  & * & * \\ * & * & * & * \end{bmatrix}$ have the following form:

<img src="graph_dense.png" width=300 align="center"> and <img src="graph_sparse.png" width=300 align="center">

* Why the second ordering is better than the first one?

In [ ]:
def graph_of(A):
    A = sp.csr_array(A)
    return [set(map(int, A.indices[A.indptr[i]:A.indptr[i + 1]])) - {i} for i in range(A.shape[0])]

def elimination_game(adj, order):
    '''Symbolic Cholesky: returns nnz(L) (with diagonal) and the list of fill edges.'''
    adj = [set(s) for s in adj]
    done = np.zeros(len(adj), bool)
    nnzL, fill = 0, []
    for v in map(int, order):
        nb = [u for u in adj[v] if not done[u]]
        nnzL += len(nb) + 1
        for a in range(len(nb)):
            for b in range(a + 1, len(nb)):
                x, y = nb[a], nb[b]
                if y not in adj[x]:
                    adj[x].add(y); adj[y].add(x); fill.append((x, y))
        done[v] = True
    return nnzL, fill

def min_degree(adj):
    '''Exact (slow, O(n^2)) minimum degree ordering via the elimination game.'''
    adj = [set(s) for s in adj]
    alive, order = set(range(len(adj))), []
    while alive:
        v = min(alive, key=lambda u: len(adj[u]))
        for x in adj[v]:
            adj[x] |= adj[v]; adj[x].discard(x); adj[x].discard(v)
        alive.remove(v); order.append(v); adj[v] = set()
    return np.array(order)

def nested_dissection_grid(n, leaf=2):
    '''Geometric nested dissection of the n x n grid (index = i*n + j): separators are numbered last.'''
    order = []
    def rec(i0, i1, j0, j1):
        if i1 <= i0 or j1 <= j0:
            return
        if (i1 - i0) * (j1 - j0) <= leaf * leaf:
            order.extend(i * n + j for i in range(i0, i1) for j in range(j0, j1)); return
        if i1 - i0 >= j1 - j0:
            m = (i0 + i1) // 2
            rec(i0, m, j0, j1); rec(m + 1, i1, j0, j1)
            order.extend(m * n + j for j in range(j0, j1))
        else:
            m = (j0 + j1) // 2
            rec(i0, i1, j0, m); rec(i0, i1, m + 1, j1)
            order.extend(i * n + m for i in range(i0, i1))
    rec(0, n, 0, n)
    return np.array(order)

In [ ]:
n = 3
adj = graph_of(lap2d(n))
orders = {'natural': np.arange(n * n),
          'nested dissection': nested_dissection_grid(n),
          'minimum degree': min_degree(adj)}
#pos = {k: (k % n, k // n) for k in range(n * n)}
pos = {k: (k % n, -(k // n)) for k in range(n * n)}                    # строка 0 сверху, как в матрице
G = nx.Graph([(i, j) for i in range(n * n) for j in adj[i] if i < j])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, order) in zip(axes, orders.items()):
    nnzL, fill = elimination_game(adj, order)
    rank = np.empty(n * n, int); rank[order] = np.arange(n * n)
    nx.draw_networkx_edges(nx.Graph(fill), pos, ax=ax, edge_color='red', alpha=0.3)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='k', width=1.5)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=70, node_color=[rank[v] for v in G.nodes], cmap='viridis')
    ax.set_title(f'{name}: {len(fill)} fill edges, nnz(L) = {nnzL}'); ax.set_aspect('equal'); ax.axis('off')
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, n*n - 1))
plt.colorbar(sm, ax=axes, label='порядок исключения (0 = первым)')
plt.show()
#fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
plt.figure(figsize=(3.5, 3.5))
A = lap2d(n)
plt.spy(A);
#for ax, (name, order) in zip(axes, orders.items()):
#    ax.spy(A[order][:, order]); plt.title(name);

- Colour shows elimination time (dark = early, yellow = late), red edges are fill-in.
- **Natural**: elimination is a front sweeping the grid, and every vertex on the front is connected to every other: band fill.
- **Nested dissection**: the separators (the yellow cross) are eliminated last, so fill stays inside small subdomains.
- **Minimum degree**: a purely local greedy rule with no global plan, and yet it is competitive.

## What can we do to minimize fill-in?

- Reordering rows and columns to minimize $\mathrm{nnz}(L) + \mathrm{nnz}(U)$ is called **fill-in minimization**.
- It is **NP-hard**: Yannakakis (1981) for the symmetric case; Rose & Tarjan (1975) for the unsymmetric (directed) case.
- But many heuristics exist:
    - **Markowitz pivoting**: order by the product of row and column nonzero counts, with a stability constraint
    - **Minimum degree** ordering (today: AMD, Amestoy, Davis & Duff 1996)
    - **(Reverse) Cuthill–McKee**: minimizes bandwidth/profile, not fill directly
    - **Nested dissection** (George 1973): recursively split the graph by a small vertex separator. Its quality depends on separator size: 1 vertex for 1D Laplacian, $\sqrt{N}$ in 2D, $N^{2/3}$ in 3D.

## Orderings at scale: 2D Laplacian

Let us compare growth of $\mathrm{nnz}(L) + \mathrm{nnz}(U)$ with $N$ for different orderings.

In [ ]:
ns = [16, 32, 64, 128]
methods = {
    'natural':           lambda A, n: lu_nnz(A)[0],
    'RCM':               lambda A, n: lu_nnz(A, perm=reverse_cuthill_mckee(sp.csr_matrix(A), symmetric_mode=True))[0],
    'COLAMD (default)':  lambda A, n: lu_nnz(A, permc_spec='COLAMD')[0],
    'MMD on A^T + A':    lambda A, n: lu_nnz(A, permc_spec='MMD_AT_PLUS_A')[0],
    'nested dissection': lambda A, n: lu_nnz(A, perm=nested_dissection_grid(n))[0],
}
res = {m: [] for m in methods}
for n in ns:
    A = lap2d(n)
    for m, f in methods.items():
        res[m].append(f(A, n))

N = np.array(ns) ** 2
plt.figure(figsize=(8, 5))
for m, v in res.items():
    plt.loglog(N, v, 'o-', label=m)
plt.loglog(N, 2 * N ** 1.5, 'k--', label=r'$N^{3/2}$')
plt.loglog(N, 3 * N * np.log2(N), 'k:', label=r'$N \log N$')
plt.xlabel('N'); plt.ylabel('nnz(L) + nnz(U)'); plt.legend(); plt.grid(True, which='both', alpha=0.3); plt.show()
for m, v in res.items():
    print(f'{m:20s}', v)

## How to formalize reduction to block arrowhead form? Separators

**Definition.** A **separator** in a graph $G$ is a set $S$ of vertices whose removal leaves at least two connected components.

A separator $S$ gives the following ordering of an $N$-vertex graph $G$:
- find $S$ whose removal leaves components $T_1, \ldots, T_k$;
- number the vertices of $S$ last: from $N - |S| + 1$ to $N$;
- recursively number the vertices of each component: $T_1$ from $1$ to $|T_1|$, $T_2$ from $|T_1| + 1$ to $|T_1| + |T_2|$, etc.;
- if a component is small enough, number it arbitrarily (or by minimum degree).

For the 2D Laplacian $A_{2D} = I \otimes A_{1D} + A_{1D} \otimes I$, $A_{1D} = \mathrm{tridiag}(-1, 2, -1)$, the separator is a middle grid line. Ordering $\alpha$ (left), $\beta$ (right), $\sigma$ (separator):


<img src='separator.png' width=300> </img>

$$
PAP^\top = \begin{bmatrix} A_{\alpha\alpha} &  & A_{\alpha\sigma} \\  & A_{\beta\beta} & A_{\beta\sigma} \\ A_{\sigma\alpha} & A_{\sigma\beta} & A_{\sigma\sigma}\end{bmatrix}
$$

So the problem of finding a **permutation** reduces to finding a **graph separator**!

In [ ]:
n = 32
A = lap2d(n); p = nested_dissection_grid(n)
_, lu_nd = lu_nnz(A, perm=p)
rank = np.empty(n * n, int); rank[p] = np.arange(n * n)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
im = ax[0].imshow(rank.reshape(n, n), cmap='viridis'); plt.colorbar(im, ax=ax[0])
ax[0].set_title('elimination order on the grid')
ax[1].spy(A[p, :][:, p], markersize=0.5); ax[1].set_title('P A P^T: recursive arrowhead')
ax[2].spy(lu_nd.L, markersize=0.5); ax[2].set_title(f'L, nnz = {lu_nd.L.nnz} (natural: {lu_nnz(A)[1].L.nnz})')
plt.show()

## Nested dissection: complexity

- Recurse on $A_{\alpha\alpha}$, $A_{\beta\beta}$; at the end the Schur complement block in place of $A_{\sigma\sigma}$ becomes dense.
- Top-level separator of a 2D grid has $n = \sqrt{N}$ vertices; its dense Cholesky costs $\mathcal{O}(n^3) = \mathcal{O}(N^{3/2})$. Lower levels form a geometric series, so the total is

$$\text{flops} = \mathcal{O}(N^{3/2}), \qquad \mathrm{nnz}(L) = \mathcal{O}(N \log N).$$

- **Planar separator theorem** (Lipton & Tarjan 1979): every planar graph has a balanced separator of size $\mathcal{O}(\sqrt{N})$, so the same bounds hold for any planar mesh.
- 3D grids: separator $N^{2/3}$, giving $\mathrm{nnz}(L) = \mathcal{O}(N^{4/3})$ and flops $= \mathcal{O}(N^2)$. This is why direct solvers struggle in 3D and iterative methods win.

### Summary
- divide and conquer: find a separator, recurse on the parts, number the separator last;
- asymptotically optimal for 2D grids; in practice beats minimum degree on large problems;
- independent subtrees give natural parallelism (elimination tree).

## Packages (2020s)

- **SuiteSparse**: CHOLMOD (supernodal Cholesky, used in MATLAB `\`, Python: `scikit-sparse`), UMFPACK (multifrontal LU), KLU, AMD/COLAMD
- **MUMPS**: MUltifrontal Massively Parallel sparse direct Solver (MPI)
- **PARDISO**: Panua PARDISO and Intel oneMKL PARDISO (Python: `pypardiso`)
- **SuperLU** / SuperLU_DIST (behind `scipy.sparse.linalg.splu`)
- **STRUMPACK**: multifrontal + low-rank (HSS / BLR) compression
- **NVIDIA cuDSS**: sparse direct solver on GPUs
- **METIS / ParMETIS / Scotch**: graph partitioning and nested dissection orderings (Python: `pymetis`)

All have C/C++/Fortran interfaces, most have Python/Julia/MATLAB bindings.

## Separators in practice

Computing good separators is **not trivial**; graph partitioning has long been an active area, often motivated by parallel computing.

- **Spectral partitioning**: eigenvectors of the graph Laplacian (below)
- **Geometric partitioning** for meshes with vertex coordinates (Miller, Teng, Thurston, Vavasis)
- **Iterative swapping**: Kernighan–Lin (1970), Fiduccia–Mattheyses (1982)
- **Breadth-first search** level structures (Lipton–Tarjan 1979)
- **Multilevel recursive bisection**: the most practical heuristic today (METIS, Karypis & Kumar 1998)

## Spectral graph partitioning

- Goes back to Miroslav Fiedler (1973), who studied algebraic connectivity of graphs.
- Split the vertices into two sets with labels $x_i \in \{+1, -1\}$. Each **cut** edge contributes $(x_i - x_j)^2 = 4$, each uncut edge $0$:

$$\mathrm{cut}(x) = \frac{1}{4}\sum_{\{i,j\} \in E} (x_i - x_j)^2 = \frac{1}{4}\, x^\top L x .$$

- **Graph Laplacian** $L = D - A_G$: $L_{ii} = \deg(i)$, $L_{ij} = -1$ if $\{i, j\} \in E$, $0$ otherwise.
- $L = BB^\top$, where $B$ is the vertex–edge incidence matrix (column $e = \{i,j\}$ has $+1$ in row $i$ and $-1$ in row $j$), so $x^\top L x = \|B^\top x\|^2 \geq 0$: $L$ is PSD.
- $Le = 0$ with $e = (1, \dots, 1)^\top$: eigenvalue $\lambda_1 = 0$.
- Balanced partition: $\sum_i x_i = x^\top e = 0$; and $\|x\|_2^2 = n$ for $\pm 1$ labels.

## Partitioning as an optimization problem

Relax the (NP-hard) integer problem to real $x$:

$$\min_{x^\top e = 0,\ \|x\|_2^2 = n} x^\top L x = n \min_{x^\top e = 0} \frac{x^\top L x}{x^\top x} = n \lambda_2,$$

by the Courant–Fischer theorem, since $e$ is the eigenvector of $\lambda_1 = 0$. The minimizer is the **Fiedler vector** $v_2$.

- **Rounding**: $\mathrm{sign}(v_2)$, or split at the **median** of $v_2$ to guarantee balance.
- **Algebraic connectivity** $\lambda_2$: $\lambda_2 > 0$ iff $G$ is connected. Cheeger inequality: $\frac{\lambda_2}{2} \leq h(G) \leq \sqrt{2 d_{\max} \lambda_2}$, where $h(G)$ is the edge expansion.
- Computing $v_2$: the power method finds the *largest* eigenvalue. We need the second *smallest*: Lanczos / LOBPCG with shift-invert or deflation, using only a few matrix-by-vector products or solves (later in the course).

In [ ]:
G = nx.karate_club_graph()                              # Zachary 1977: the club really split in two
L = nx.laplacian_matrix(G, weight=None).astype(float).toarray()   # weight=None: unweighted Laplacian
lam, V = np.linalg.eigh(L)
f = V[:, 1]
plt.figure()
plt.spy(L)
print('3 smallest eigenvalues:', lam[:3].round(4), ' sum of Fiedler vector =', f.sum().round(12))

truth = np.array([G.nodes[v]['club'] == 'Officer' for v in G])
pred = f > 0
acc = max(np.mean(pred == truth), np.mean(pred != truth))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
o = np.argsort(f)
ax[0].bar(range(len(f)), f[o], color=np.where(truth[o], 'tab:orange', 'tab:blue'))
ax[0].axhline(0, color='k'); ax[0].set_title('sorted Fiedler vector, colour = real faction after the split')
nx.draw_networkx(G, nx.spring_layout(G, seed=1), ax=ax[1], node_color=np.sign(f), cmap='coolwarm')
ax[1].set_title(f'sign(v2) agrees with the real split for {acc:.0%} of members'); ax[1].axis('off')
plt.show()

## From Fiedler vector to a vertex separator

$\mathrm{sign}(v_2)$ gives an **edge** cut, but nested dissection needs a **vertex** separator:
- option 1: threshold $\tau > 0$; vertices with $v_i < -\tau$, $|v_i| \leq \tau$, $v_i > \tau$ form part A, the separator, and part B. $\tau$ controls the separator size, but you must check that it really separates;
- option 2 (used below): take the vertices of one part that have neighbours in the other part (the smaller of the two boundaries). This is always a valid separator.

On a $40 \times 20$ grid, the Fiedler vector is $\cos(\pi (x + 1/2) / 40)$: spectral partitioning *automatically* cuts across the short side.

In [ ]:
def fiedler_vector(Adj):
    L = laplacian(sp.csr_matrix(Adj, dtype=float))
    if L.shape[0] <= 400:
        return np.linalg.eigh(L.toarray())[1][:, 1]
    lam, V = spla.eigsh(L, k=2, sigma=-1e-3, which='LM')   # shift-invert near 0
    return V[:, np.argsort(lam)[1]]

def vertex_separator(Adj, f):
    '''Median split of the Fiedler vector; the smaller boundary becomes a vertex separator.'''
    Adj = sp.csr_matrix(Adj)
    part = f > np.median(f)
    bA = np.flatnonzero(part & (Adj @ (~part).astype(float) > 0))
    bB = np.flatnonzero(~part & (Adj @ part.astype(float) > 0))
    s = bA if len(bA) <= len(bB) else bB
    mask = np.zeros(len(f), bool); mask[s] = True
    return np.flatnonzero(part & ~mask), np.flatnonzero(~part & ~mask), s

Gg = nx.grid_2d_graph(40, 20)
xy = np.array(list(Gg.nodes))
Adj = nx.adjacency_matrix(Gg, weight=None)
f = fiedler_vector(Adj)
a, b, s = vertex_separator(Adj, f)
lab = np.zeros(len(f)); lab[b] = 1.0; lab[s] = 0.5

def on_grid(v):
    M = np.zeros((40, 20)); M[xy[:, 0], xy[:, 1]] = v
    return M.T

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
im = ax[0].imshow(on_grid(f), cmap='coolwarm'); plt.colorbar(im, ax=ax[0]); ax[0].set_title('Fiedler vector of a 40 x 20 grid')
ax[1].imshow(on_grid(lab), cmap='coolwarm'); ax[1].set_title(f'parts: {len(a)} / {len(b)}, separator: {len(s)} vertices')
plt.show()

## Practical problems

**1. Computing bisections recursively with eigenvectors is expensive.** In practice, **multilevel bisection** (METIS) has 3 phases:
- *coarsening*: merge vertices (heavy-edge matching), giving graphs $G_1, G_2, \ldots, G_m$ of decreasing size;
- *initial partition* of the small graph $G_m$ (spectral, greedy, ...);
- *uncoarsening*: project the split from $G_k$ to $G_{k-1}$ and refine it locally (Kernighan–Lin / Fiduccia–Mattheyses).

**2. Once the permutation is known, elimination must use fast kernels.**
- Columns of $L$ with identical sparsity structure form **supernodes**: dense blocks of $L$.
- Supernodal and multifrontal methods operate on these dense blocks with BLAS-3 (GEMM), reaching a large fraction of peak flops.
- The *elimination tree* exposes the parallelism: independent subtrees = independent subdomains of nested dissection.

Details: *State-of-The-Art Sparse Direct Solvers* ([arXiv:1907.05309](https://arxiv.org/abs/1907.05309)).

## Take home message

- The inverse of a sparse matrix is dense, but the factors $L$, $U$ can be sparse; ordering decides everything.
- Graph view: elimination = making the neighbours a clique; fill = paths through earlier-eliminated vertices.
- Fill minimization is NP-hard; heuristics are Markowitz, (approximate) minimum degree, RCM (bandwidth, not fill!), nested dissection.
- Separators turn the matrix into a recursive block arrowhead; 2D: $\mathcal{O}(N\log N)$ memory, $\mathcal{O}(N^{3/2})$ flops; 3D: $\mathcal{O}(N^{4/3})$ and $\mathcal{O}(N^2)$.
- Fiedler vector = relaxed minimum balanced cut; it also finds real communities.


## Plan for the next lecture

- Basic iterative methods for solving large linear systems
- Convergence
- Acceleration

# Questions?